# Examples

In [1]:
import numpy as np
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings("ignore")

## Example 1: DRlm - Regression

In [2]:
from linear import Regression, Classification

### Data Generating Process

In [3]:
np.random.seed(0)  # For reproducibility
# number of groups
L = 2
# dimension
p = 100

# mean vector for source
mean_source = np.zeros(p)

# covariance matrix for source
def A1gen(rho, p):
    A1 = np.zeros((p, p))
    for i in range(p):
        for j in range(p):
            A1[i, j] = rho ** abs(i - j)
    return A1

cov_source = A1gen(0.6, p)

# 1st group's source data
n1 = 100
X1 = multivariate_normal.rvs(mean=mean_source, cov=cov_source, size=n1)
b1 = np.zeros(p)
b1[0:5] = np.arange(1, 6) / 20
b1[97:100] = [0.5, -0.5, -0.5]
Y1 = X1 @ b1 + np.random.normal(size=n1)

# 2nd group's source data
n2 = 100
X2 = multivariate_normal.rvs(mean=mean_source, cov=cov_source, size=n2)
b2 = np.zeros(p)
b2[5:10] = np.arange(1, 6) / 20
b2[97:100] = 0.5 * np.array([0.5, -0.5, -0.5])
Y2 = X2 @ b2 + np.random.normal(size=n2)

# Target Data, covariate shift
n0 = 100
mean0 = np.zeros(p) + 0.1
cov0 = cov_source.copy()

# diagonal elements
for i in range(p):
    cov0[i, i] = 1.5

# first 5x5 block off-diagonal
for i in range(5):
    for j in range(5):
        if i != j:
            cov0[i, j] = 0.9

# last 2x2 block off-diagonal (indices 98~100 in R = 97~99 in Python)
for i in range(98, 100):
    for j in range(98, 100):
        if i != j:
            cov0[i, j] = 0.9

X0 = multivariate_normal.rvs(mean=mean0, cov=cov0, size=n0)

Xlist = [X1, X2]
ylist = [Y1, Y2]


In [4]:
# dimension p=100
loading_mat = np.zeros((100, 2))
loading_mat[95:100, 0] = 0.4  
loading_mat[98:100, 1] = 0.8

loading_mat = loading_mat.T


### Implementation

In [5]:
reg = Regression(f_learner='high_d', verbose=True)
reg.fit(Xlist, ylist, loading_mat, X0=X0)
reg.infer(M=200, alpha=0.05, alpha_thres=0.01)

## time cost: 6.6s

Argument 'loading_intercept' set to False because intercept is False
start fitting-----
======> Bias Correction for initial estimators....
---> Computing for loading (1/2)...
The projection direction is identified at xi = 0.017807 at step = 7.0
---> Computing for loading (2/2)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (1/2)...
The projection direction is identified at xi = 0.026710 at step = 6.0
---> Computing for loading (2/2)...
The projection direction is identified at xi = 0.026710 at step = 6.0
======> Bias Correction for matrix Gamma....
---> Computing for loading (1/1)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (1/1)...
The projection direction is identified at xi = 0.026710 at step = 6.0
---> Computing for loading (1/1)...
The projection direction is identified at xi = 0.026710 at step = 6.0
---> Computing for loading (1/1)...
The projection direction is identified

### Results

In [6]:
reg.summary()

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.1865   0.8135

Fitted Plug-in Estimations:

index     |        1        2
coef_     |  -0.1112  -0.2564

Fitted Debiased Estimations:

index     |        1        2
coef_     |  -0.0947  -0.4979

Confidence Intervals for each coefficient:

index     |              1              2
CI        | (-0.2751,0.0857) (-0.8399,-0.1559)



In [7]:
reg.summary(index=[2])

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.1865   0.8135

Fitted Plug-in Estimations:

index     |        2
coef_     |  -0.2564

Fitted Debiased Estimations:

index     |        2
coef_     |  -0.4979

Confidence Intervals for each coefficient:

index     |              2
CI        | (-0.8399,-0.1559)



In [8]:
reg.predict()

array([ 0.57151319, -0.02928899, -0.06033991,  0.11612113, -0.68601212,
       -0.53553375, -0.21904787,  0.13833815, -0.76199818,  1.1203026 ,
        0.44810204,  0.57437405, -0.36665777, -0.53966998,  0.34693913,
       -0.15112281,  0.1967401 , -0.31707418, -0.81371427, -0.44283385,
       -0.30694343,  0.39382992,  1.04627472,  0.24925858, -0.52190778,
        0.91319591,  0.0698671 ,  0.04021008,  0.82834396,  0.02960527,
        0.01608838, -0.22463264,  0.32442729,  0.04313943, -0.81462286,
       -0.17720282, -0.35866089, -0.54523625, -0.37470143, -0.28446907,
       -0.00845137, -0.72892044,  0.63640322,  1.28434837,  0.00191001,
        0.04363812, -0.72227019, -0.18187351,  0.50318242,  0.28319007,
        0.19396803,  0.1015529 ,  0.56790442,  0.05540394, -1.04799782,
       -0.99332377,  0.3305164 ,  1.33258481, -1.28313494,  0.341081  ,
        0.33124495,  0.20624512, -0.02379628,  0.55200241,  0.39877499,
       -0.51265527, -0.82236677,  0.45091854, -0.19443793,  0.00

## Example 2: DRlm - Classification

### Data Generating Process

In [9]:
def softmax(x):
    """
    Input: dim:n*(C-1)
    Output: softmax probabilities, dim:n*C
    
    """
    
    x_max = np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

np.random.seed(123)
n = 100; p = 5; L = 2; N = 1000
K = 2 # Number of classes
Xlist = [np.random.normal(0, 1, (n, p)) for _ in range(L)]
X0 = np.random.normal(0.1, 1, (N, p))
beta_list = [np.column_stack((np.zeros(p),np.random.normal(0, 0.25, (p,K)))) for _ in range(L)]
logits_list = [
    X.dot(beta) - np.mean(X.dot(beta))
    for X, beta in zip(Xlist, beta_list)
]
probs_list = [softmax(logits) for logits in logits_list]
ylist = [np.array([np.random.multinomial(1, probs[i, :]).tolist().index(1) for i in range(n)]) for probs in probs_list]


### Implementation

In [10]:
cc = Classification(f_learner='linear', w_learner='linear')
cc.fit(Xlist,ylist,X0)
cc.infer()

## time cost: 4.8s

### Results

In [11]:
cc.summary()

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.3583   0.6417

Fitted Coefficients:

Class 1 coefficients:
index     |        1        2        3        4        5
coef_     |   0.2330   0.0147  -0.1428   0.2283   0.7163

Class 2 coefficients:
index     |        1        2        3        4        5
coef_     |  -0.2345   0.6903   0.2463  -0.2958   0.2633

Confidence Intervals for each coefficient:

Class 1 Confidence Intervals:
index     |              1              2              3              4              5
CIs       | (-1.233,3.995) (-3.886,8.197) (-1.905,1.496) (-1.508,4.038) (-0.243,7.694)

Class 2 Confidence Intervals:
index     |              1              2              3              4              5
CIs       | (-1.764,1.550) (-1.791,13.050) (-0.877,3.586) (-2.148,1.260) (-2.278,6.150)



In [12]:
cc.summary(
    index = [3,5], class_index=2
)

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.3583   0.6417

Fitted Coefficients:

Class 2 coefficients:
index     |        3        5
coef_     |   0.2463   0.2633

Confidence Intervals for each coefficient:

Class 2 Confidence Intervals:
index     |              3              5
CIs       | (-0.877,3.586) (-2.278,6.150)



In [13]:
cc.predict_proba()

{'pred_proba': array([[0.18597781, 0.17658149, 0.63744071],
        [0.44065281, 0.12374689, 0.43560029],
        [0.25250153, 0.46645683, 0.28104164],
        ...,
        [0.18052818, 0.28350621, 0.53596561],
        [0.19853889, 0.60943075, 0.19203037],
        [0.12645667, 0.12713534, 0.74640799]])}

In [14]:
cc.predict()

array([2, 0, 1, 2, 2, 2, 2, 1, 0, 1, 1, 2, 1, 1, 0, 1, 0, 1, 2, 2, 2, 1,
       2, 2, 1, 2, 0, 2, 1, 0, 2, 2, 0, 2, 0, 1, 0, 1, 2, 0, 0, 2, 1, 1,
       2, 2, 1, 1, 1, 1, 2, 2, 1, 2, 1, 2, 0, 0, 1, 2, 2, 2, 1, 2, 2, 2,
       0, 1, 2, 2, 0, 1, 1, 2, 1, 1, 1, 0, 2, 1, 1, 0, 2, 1, 2, 2, 1, 0,
       2, 2, 1, 2, 2, 1, 1, 1, 2, 2, 0, 1, 2, 0, 0, 1, 2, 0, 2, 1, 0, 0,
       1, 1, 0, 0, 2, 2, 2, 1, 1, 2, 2, 1, 1, 1, 0, 0, 2, 2, 2, 2, 1, 2,
       0, 2, 2, 1, 1, 1, 0, 2, 0, 2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 0, 1, 2,
       2, 1, 1, 2, 1, 0, 1, 2, 2, 1, 0, 2, 2, 1, 0, 1, 0, 2, 1, 1, 0, 2,
       2, 2, 2, 1, 1, 0, 1, 2, 1, 0, 0, 2, 0, 0, 2, 2, 0, 1, 2, 2, 2, 1,
       1, 0, 0, 2, 2, 1, 1, 1, 1, 2, 1, 1, 1, 0, 2, 1, 1, 2, 0, 2, 0, 1,
       1, 0, 2, 1, 2, 1, 2, 1, 2, 2, 2, 1, 2, 1, 0, 1, 2, 2, 1, 1, 1, 2,
       0, 2, 0, 2, 1, 1, 0, 2, 1, 1, 0, 1, 1, 2, 1, 1, 1, 2, 2, 2, 0, 0,
       0, 1, 1, 1, 1, 1, 0, 2, 0, 2, 1, 2, 2, 0, 2, 2, 2, 1, 0, 2, 1, 1,
       1, 0, 2, 2, 2, 2, 0, 1, 2, 2, 2, 1, 2, 1, 2,

## Example 3: DRoL

### Data Generating Process

In [15]:
from data import *
from nonlinear import Regression
data = DataContainerSimu1(n=2000, N=20000)
data.generate_funcs_list(L=2, seed=0)
data.generate_data()

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target

### Implementation

In [16]:
drol = Regression(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0)

## time cost: 8.7s

### Results

In [17]:
drol.weight_

array([0.29488955, 0.70511045])

### Prediction

In [18]:
drol.predict()

array([-1.31915748,  0.17683433,  0.7920894 , ..., -0.50509355,
        3.91434165, -2.28899691])